# Ethical Considerations in Machine Learning

This notebook explores key ethical issues in machine learning including:
- Bias and fairness
- Privacy concerns
- Transparency and explainability
- Accountability and governance
- Real-world case studies

These considerations are crucial as machine learning systems increasingly impact critical aspects of human life.

## 1. Import Required Libraries

In [ ]:
# Core data science and ML libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# For explainability
import shap

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("deep")

# Import optional fairness-specific libraries (with error handling)
try:
    # Fairlearn for fairness metrics and mitigation
    import fairlearn
    from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
    from fairlearn.reductions import ExponentiatedGradient, DemographicParity
    FAIRLEARN_AVAILABLE = True
except ImportError:
    print("Fairlearn library not available. Some fairness metrics won't be demonstrated.")
    FAIRLEARN_AVAILABLE = False

try:
    # AI Fairness 360 for bias detection and mitigation
    import aif360
    from aif360.datasets import BinaryLabelDataset
    from aif360.metrics import BinaryLabelDatasetMetric
    AIF360_AVAILABLE = True
except ImportError:
    print("AI Fairness 360 library not available. Some bias detection methods won't be demonstrated.")
    AIF360_AVAILABLE = False

## 2. Understanding Bias in Data and Algorithms

Bias can enter machine learning systems at multiple points:
1. **Data collection**: Underrepresentation of certain groups
2. **Data labeling**: Human prejudices in labeling
3. **Feature selection**: Choosing features correlated with protected attributes
4. **Algorithm design**: Optimizing for metrics that don't account for fairness
5. **Deployment**: Using models in contexts different from training

Let's create a simple synthetic dataset to demonstrate bias:

In [ ]:
# Create a synthetic dataset with inherent bias
np.random.seed(42)

# Sample size
n = 1000

# Create features
age = np.random.normal(40, 10, n)
income = np.random.normal(50000, 15000, n)
# Create gender as binary feature (0: female, 1: male)
gender = np.random.binomial(1, 0.5, n)

# Introduce bias: males tend to have higher incomes
income = income + gender * 10000

# Create a biased feature that's correlated with gender
years_experience = np.random.normal(10, 3, n) + gender * 2

# Create education level (0-6 representing education levels)
education = np.random.randint(0, 7, n)

# Create biased target: loan approval 
# Males and those with higher incomes have higher approval rates
approval_probability = 0.3 + 0.4 * (income > 50000) + 0.2 * gender
loan_approved = np.random.binomial(1, approval_probability, n)

# Create DataFrame
data = pd.DataFrame({
    'Age': age,
    'Income': income,
    'Gender': gender,
    'YearsExperience': years_experience,
    'Education': education,
    'LoanApproved': loan_approved
})

# Map gender to string categories
data['GenderCategory'] = data['Gender'].map({0: 'Female', 1: 'Male'})

# Display first few rows
data.head()

In [ ]:
# Analyze distribution of loan approvals based on gender
gender_approval = data.groupby('GenderCategory')['LoanApproved'].mean()

plt.figure(figsize=(10, 6))
sns.barplot(x=gender_approval.index, y=gender_approval.values)
plt.title('Loan Approval Rate by Gender', fontsize=15)
plt.ylabel('Approval Rate')
plt.ylim(0, 1)

for i, rate in enumerate(gender_approval):
    plt.text(i, rate + 0.02, f'{rate:.2f}', ha='center')

plt.show()

# Show approval rates across both gender and income levels
plt.figure(figsize=(12, 6))
data['IncomeGroup'] = pd.qcut(data['Income'], 4, labels=['Low', 'Medium-Low', 'Medium-High', 'High'])
approval_by_gender_income = data.groupby(['GenderCategory', 'IncomeGroup'])['LoanApproved'].mean().unstack()

approval_by_gender_income.plot(kind='bar', figsize=(12, 6))
plt.title('Loan Approval Rate by Gender and Income Level', fontsize=15)
plt.ylabel('Approval Rate')
plt.ylim(0, 1)
plt.legend(title='Income Level')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Print exact numbers
print("Loan approval rates:")
print(approval_by_gender_income)

### Training a biased model

Now let's see how a machine learning model might perpetuate or amplify this bias:

In [ ]:
# Prepare data for modeling
X = data[['Age', 'Income', 'YearsExperience', 'Education']]  # Not directly using gender
y = data['LoanApproved']
sensitive_feature = data['Gender']  # For fairness evaluation

# Split data
X_train, X_test, y_train, y_test, gender_train, gender_test = train_test_split(
    X, y, sensitive_feature, test_size=0.3, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train a logistic regression model
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Evaluate overall performance
print("Overall model accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Evaluate predictions by gender
gender_test = gender_test.reset_index(drop=True)
results_df = pd.DataFrame({
    'Gender': gender_test,
    'True': y_test.reset_index(drop=True),
    'Predicted': y_pred
})

gender_accuracy = results_df.groupby('Gender').apply(lambda g: accuracy_score(g['True'], g['Predicted']))
gender_approval_rate = results_df.groupby('Gender')['Predicted'].mean()

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
gender_accuracy.plot(kind='bar', color=['#FF9999', '#66B2FF'])
plt.title('Model Accuracy by Gender')
plt.ylabel('Accuracy')
plt.xticks([0, 1], ['Female', 'Male'], rotation=0)
for i, acc in enumerate(gender_accuracy):
    plt.text(i, acc - 0.05, f'{acc:.3f}', ha='center', color='black', fontweight='bold')

plt.subplot(1, 2, 2)
gender_approval_rate.plot(kind='bar', color=['#FF9999', '#66B2FF'])
plt.title('Predicted Approval Rate by Gender')
plt.ylabel('Approval Rate')
plt.xticks([0, 1], ['Female', 'Male'], rotation=0)
for i, rate in enumerate(gender_approval_rate):
    plt.text(i, rate - 0.05, f'{rate:.3f}', ha='center', color='black', fontweight='bold')

plt.tight_layout()
plt.show()

### Feature Importance Analysis

Let's examine which features the model is relying on most heavily:

In [ ]:
# Train a decision tree for better interpretability
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_model.fit(X_train, y_train)

# Get feature importance
feature_importance = dt_model.feature_importances_
feature_names = X.columns

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

# Visualize feature importance
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Feature Importance in Loan Approval Decision', fontsize=15)
plt.tight_layout()
plt.show()

# Print exact values
print("Feature importance:")
print(importance_df)

## 3. Fairness Metrics and Evaluation

Different metrics can be used to evaluate fairness in machine learning systems:

1. **Demographic Parity**: Equal prediction rates across protected groups
2. **Equal Opportunity**: Equal true positive rates across protected groups
3. **Equalized Odds**: Equal true positive and false positive rates across groups
4. **Disparate Impact**: Ratio of prediction rates between advantaged and disadvantaged groups

Let's evaluate our model using these metrics:

In [ ]:
# Calculate basic fairness metrics manually

# Group predictions by gender
female_preds = y_pred[gender_test == 0]
male_preds = y_pred[gender_test == 1]
female_true = y_test.reset_index(drop=True)[gender_test == 0]
male_true = y_test.reset_index(drop=True)[gender_test == 1]

# Calculate demographic parity (difference in approval rates)
female_approval_rate = female_preds.mean()
male_approval_rate = male_preds.mean()
demographic_parity_diff = male_approval_rate - female_approval_rate

# Calculate true positive rates (equal opportunity)
female_tpr = np.sum((female_preds == 1) & (female_true == 1)) / np.sum(female_true == 1)
male_tpr = np.sum((male_preds == 1) & (male_true == 1)) / np.sum(male_true == 1)
equal_opportunity_diff = male_tpr - female_tpr

# Calculate false positive rates (equalized odds also includes FPR)
female_fpr = np.sum((female_preds == 1) & (female_true == 0)) / np.sum(female_true == 0)
male_fpr = np.sum((male_preds == 1) & (male_true == 0)) / np.sum(male_true == 0)
fpr_diff = male_fpr - female_fpr

# Print metrics
print(f"Demographic Parity Difference: {demographic_parity_diff:.4f}")
print(f"Equal Opportunity Difference: {equal_opportunity_diff:.4f}")
print(f"False Positive Rate Difference: {fpr_diff:.4f}")

# Calculate Disparate Impact
disparate_impact = female_approval_rate / male_approval_rate
print(f"Disparate Impact: {disparate_impact:.4f}")
print("(Values closer to 1.0 indicate more fairness)")

# Visualize fairness metrics
metrics = ['Demographic\nParity Diff', 'Equal Opportunity\nDiff', 'FPR\nDiff']
values = [demographic_parity_diff, equal_opportunity_diff, fpr_diff]

plt.figure(figsize=(12, 6))
bars = plt.bar(metrics, values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])

# Color bars based on fairness (green for fair, red for unfair)
for i, bar in enumerate(bars):
    if abs(values[i]) < 0.05:
        bar.set_color('#4CAF50')  # Green if close to 0
    elif abs(values[i]) < 0.1:
        bar.set_color('#FFC107')  # Yellow if moderate
    else:
        bar.set_color('#F44336')  # Red if large difference

plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.title('Fairness Metrics Evaluation', fontsize=15)
plt.ylabel('Difference (Male - Female)')

# Add value labels
for i, v in enumerate(values):
    plt.text(i, v + 0.01 if v > 0 else v - 0.03, f'{v:.3f}', 
             ha='center', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Fairness-aware modeling (optional, requires Fairlearn)

Let's implement a fairness-aware approach to mitigate the bias:

In [ ]:
# This cell will only run if Fairlearn is available
if FAIRLEARN_AVAILABLE:
    # Constraint for demographic parity
    constraint = DemographicParity()
    
    # Create a mitigated model using exponentiated gradient
    mitigator = ExponentiatedGradient(
        LogisticRegression(random_state=42),
        constraint=constraint
    )
    
    # Fit the mitigator
    mitigator.fit(X_train_scaled, y_train, sensitive_features=gender_train)
    
    # Make predictions
    y_pred_mitigated = mitigator.predict(X_test_scaled)
    
    # Calculate metrics again
    female_preds_mit = y_pred_mitigated[gender_test == 0]
    male_preds_mit = y_pred_mitigated[gender_test == 1]
    
    # Calculate demographic parity
    female_approval_rate_mit = female_preds_mit.mean()
    male_approval_rate_mit = male_preds_mit.mean()
    demographic_parity_diff_mit = male_approval_rate_mit - female_approval_rate_mit
    
    # Calculate disparate impact
    disparate_impact_mit = female_approval_rate_mit / male_approval_rate_mit
    
    # Display comparison
    print("Before mitigation:")
    print(f"  Female approval rate: {female_approval_rate:.4f}")
    print(f"  Male approval rate: {male_approval_rate:.4f}")
    print(f"  Demographic parity difference: {demographic_parity_diff:.4f}")
    print(f"  Disparate impact: {disparate_impact:.4f}")
    
    print("\nAfter mitigation:")
    print(f"  Female approval rate: {female_approval_rate_mit:.4f}")
    print(f"  Male approval rate: {male_approval_rate_mit:.4f}")
    print(f"  Demographic parity difference: {demographic_parity_diff_mit:.4f}")
    print(f"  Disparate impact: {disparate_impact_mit:.4f}")
    
    # Calculate overall accuracy of mitigated model
    accuracy_mitigated = accuracy_score(y_test, y_pred_mitigated)
    print(f"\nOriginal model accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Mitigated model accuracy: {accuracy_mitigated:.4f}")
    
    # Visualize comparison
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    
    # Original model approval rates
    gender_labels = ['Female', 'Male']
    original_rates = [female_approval_rate, male_approval_rate]
    mitigated_rates = [female_approval_rate_mit, male_approval_rate_mit]
    
    x = np.arange(len(gender_labels))
    width = 0.35
    
    ax[0].bar(x - width/2, original_rates, width, label='Original Model', color='#FF9999')
    ax[0].bar(x + width/2, mitigated_rates, width, label='Mitigated Model', color='#99CCFF')
    
    ax[0].set_xlabel('Gender')
    ax[0].set_ylabel('Approval Rate')
    ax[0].set_title('Approval Rates Before and After Mitigation')
    ax[0].set_xticks(x)
    ax[0].set_xticklabels(gender_labels)
    ax[0].legend()
    
    # Accuracy and fairness tradeoff
    metrics = ['Accuracy', 'Demographic\nParity Diff']
    original_metrics = [accuracy_score(y_test, y_pred), abs(demographic_parity_diff)]
    mitigated_metrics = [accuracy_mitigated, abs(demographic_parity_diff_mit)]
    
    ax[1].bar(x - width/2, original_metrics, width, label='Original Model', color='#FF9999')
    ax[1].bar(x + width/2, mitigated_metrics, width, label='Mitigated Model', color='#99CCFF')
    
    ax[1].set_xlabel('Metric')
    ax[1].set_ylabel('Value')
    ax[1].set_title('Accuracy vs. Fairness Trade-off')
    ax[1].set_xticks(x)
    ax[1].set_xticklabels(metrics)
    ax[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Fairlearn library not available. Install it with 'pip install fairlearn' to run this example.")

## 4. Privacy Concerns and Data Protection

Privacy is another important ethical consideration in ML. Here are some techniques for privacy-preserving ML:

1. **Data anonymization**
2. **Differential privacy**
3. **Federated learning**
4. **Secure multi-party computation**

Let's demonstrate some simple data anonymization techniques:

In [ ]:
# Create a copy of our dataset with more personal features
personal_data = data.copy()
personal_data['Name'] = ['Person_' + str(i) for i in range(len(personal_data))]
personal_data['SSN'] = ['123-45-' + str(1000 + i) for i in range(len(personal_data))]
personal_data['Address'] = ['123 Main St, Apt ' + str(i) for i in range(len(personal_data))]
personal_data['PhoneNumber'] = ['(555) 123-' + str(1000 + i).zfill(4) for i in range(len(personal_data))]

# Display original personal data
print("Original data with personal identifiers:")
print(personal_data[['Name', 'SSN', 'Address', 'PhoneNumber', 'Age', 'Income', 'Gender']].head())

# Simple anonymization - remove direct identifiers
anonymized_data = personal_data.drop(columns=['Name', 'SSN', 'Address', 'PhoneNumber'])

# Generalization - bin age into age groups
anonymized_data['AgeGroup'] = pd.cut(anonymized_data['Age'], 
                                    bins=[0, 25, 35, 45, 55, 65, 100],
                                    labels=['<25', '26-35', '36-45', '46-55', '56-65', '65+'])

# Income binning
anonymized_data['IncomeGroup'] = pd.qcut(anonymized_data['Income'], 5, 
                                        labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

# Apply k-anonymity concept (simplistic version)
# Grouping by AgeGroup, Gender, and IncomeGroup, ensuring groups have at least 5 members
group_counts = anonymized_data.groupby(['AgeGroup', 'Gender', 'IncomeGroup']).size()
k_value = 5
small_groups = group_counts[group_counts < k_value]

# Show small groups (privacy risk)
print(f"\nGroups with fewer than {k_value} members (privacy risk):")
if len(small_groups) > 0:
    print(small_groups)
else:
    print("No small groups found.")

# Display anonymized data
print("\nAnonymized data:")
print(anonymized_data[['AgeGroup', 'IncomeGroup', 'Gender', 'LoanApproved']].head())

### Differential Privacy Example 

Differential privacy adds noise to data or computations to protect individual privacy while maintaining overall statistical validity.

In [ ]:
# Simple example of differential privacy on aggregate statistics
np.random.seed(42)

# Helper function to add Laplace noise for differential privacy
def add_laplace_noise(value, sensitivity, epsilon):
    """
    Add Laplace noise to value for differential privacy
    
    Parameters:
    - value: The true value to protect
    - sensitivity: How much the value can change with one record (max influence)
    - epsilon: Privacy parameter (lower = more private, more noise)
    
    Returns:
    - The value with noise added
    """
    scale = sensitivity / epsilon
    noise = np.random.laplace(0, scale)
    return value + noise

# Get some true statistics from our data
true_mean_income = data['Income'].mean()
true_mean_age = data['Age'].mean()
true_approval_rate = data['LoanApproved'].mean()

# Define privacy parameter (lower = more privacy, more noise)
epsilon_values = [10, 1, 0.1, 0.01]
sensitivity = 1  # Assuming normalized sensitivity

# Generate differentially private statistics with different privacy levels
results = []
for epsilon in epsilon_values:
    # Run multiple trials for each epsilon to show variation
    noisy_income_trials = [add_laplace_noise(true_mean_income, true_mean_income/10, epsilon) 
                          for _ in range(10)]
    noisy_age_trials = [add_laplace_noise(true_mean_age, 1, epsilon) 
                       for _ in range(10)]
    noisy_approval_trials = [add_laplace_noise(true_approval_rate, 0.01, epsilon) 
                            for _ in range(10)]
    
    results.append({
        'epsilon': epsilon,
        'income_mean': np.mean(noisy_income_trials),
        'income_std': np.std(noisy_income_trials),
        'age_mean': np.mean(noisy_age_trials),
        'age_std': np.std(noisy_age_trials),
        'approval_mean': np.mean(noisy_approval_trials),
        'approval_std': np.std(noisy_approval_trials)
    })

# Display results
results_df = pd.DataFrame(results)
print("Differentially Private Statistics:")
print(results_df)
print(f"\nTrue mean income: {true_mean_income:.2f}")
print(f"True mean age: {true_mean_age:.2f}")
print(f"True approval rate: {true_approval_rate:.4f}")

# Visualize the privacy-utility trade-off
plt.figure(figsize=(15, 5))

# Income
plt.subplot(1, 3, 1)
plt.errorbar(results_df['epsilon'], results_df['income_mean'], 
             yerr=results_df['income_std'], fmt='o-', capsize=5)
plt.axhline(y=true_mean_income, color='g', linestyle='--', label='True Value')
plt.xscale('log')
plt.xlabel('Epsilon (lower = more privacy)')
plt.ylabel('Mean Income')
plt.title('Income: Privacy vs. Accuracy')
plt.legend()

# Age
plt.subplot(1, 3, 2)
plt.errorbar(results_df['epsilon'], results_df['age_mean'], 
             yerr=results_df['age_std'], fmt='o-', capsize=5)
plt.axhline(y=true_mean_age, color='g', linestyle='--', label='True Value')
plt.xscale('log')
plt.xlabel('Epsilon (lower = more privacy)')
plt.ylabel('Mean Age')
plt.title('Age: Privacy vs. Accuracy')
plt.legend()

# Approval rate
plt.subplot(1, 3, 3)
plt.errorbar(results_df['epsilon'], results_df['approval_mean'], 
             yerr=results_df['approval_std'], fmt='o-', capsize=5)
plt.axhline(y=true_approval_rate, color='g', linestyle='--', label='True Value')
plt.xscale('log')
plt.xlabel('Epsilon (lower = more privacy)')
plt.ylabel('Approval Rate')
plt.title('Approval Rate: Privacy vs. Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

### Federated Learning Concept

Federated learning allows training models across multiple devices/servers without exchanging the raw data, keeping private data local while still benefiting from collaborative learning.

Below is a simplified conceptual example of federated learning:

In [ ]:
# Simulate federated learning with our dataset (conceptual example)

# Split data to simulate multiple institutions/users
np.random.seed(42)
institution_ids = np.random.randint(1, 4, size=len(data))  # 3 different institutions
data['Institution'] = institution_ids

# Function to train local models and aggregate weights
def federated_learning_simulation(data, num_institutions, num_rounds=3):
    """Simulates a federated learning process"""
    
    # Split data by institution
    institutions = {}
    for i in range(1, num_institutions + 1):
        institutions[i] = data[data['Institution'] == i]
        print(f"Institution {i} has {len(institutions[i])} records")
    
    # Track centralized vs federated performance
    centralized_accuracy = []
    federated_accuracy = []
    
    # For demonstration, use a simple model
    for round_num in range(num_rounds):
        print(f"\nRound {round_num + 1}:")
        
        # Train local models
        local_models = {}
        for inst_id, inst_data in institutions.items():
            # Prepare local data
            X_local = inst_data[['Age', 'Income', 'YearsExperience', 'Education']]
            y_local = inst_data['LoanApproved']
            
            # Train local model
            local_model = LogisticRegression(random_state=42)
            local_model.fit(X_local, y_local)
            local_models[inst_id] = local_model
            
            print(f"  Institution {inst_id} local model coefficients: {local_model.coef_}")
        
        # Simplified weight averaging for model aggregation
        # In a real FL system, this would be more sophisticated
        aggregated_coeffs = np.zeros_like(local_models[1].coef_)
        aggregated_intercept = 0
        
        for inst_id, model in local_models.items():
            # Weight by institution size
            weight = len(institutions[inst_id]) / len(data)
            aggregated_coeffs += model.coef_ * weight
            aggregated_intercept += model.intercept_[0] * weight
        
        print(f"  Aggregated model coefficients: {aggregated_coeffs}")
        
        # Create aggregated model
        aggregated_model = LogisticRegression(random_state=42)
        aggregated_model.fit(data[['Age', 'Income', 'YearsExperience', 'Education']], data['LoanApproved'])
        aggregated_model.coef_ = aggregated_coeffs
        aggregated_model.intercept_ = np.array([aggregated_intercept])
        
        # Evaluate federated model
        X_all = data[['Age', 'Income', 'YearsExperience', 'Education']]
        y_all = data['LoanApproved']
        fed_acc = aggregated_model.score(X_all, y_all)
        federated_accuracy.append(fed_acc)
        
        # For comparison: centralized model (trained on all data)
        central_model = LogisticRegression(random_state=42)
        central_model.fit(X_all, y_all)
        central_acc = central_model.score(X_all, y_all)
        centralized_accuracy.append(central_acc)
        
        print(f"  Federated model accuracy: {fed_acc:.4f}")
        print(f"  Centralized model accuracy: {central_acc:.4f}")
    
    return centralized_accuracy, federated_accuracy

# Run federated learning simulation
centralized_acc, federated_acc = federated_learning_simulation(data, num_institutions=3)

# Visualize results
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(centralized_acc) + 1), centralized_acc, 'bo-', label='Centralized Training')
plt.plot(range(1, len(federated_acc) + 1), federated_acc, 'ro-', label='Federated Learning')
plt.xlabel('Training Round')
plt.ylabel('Model Accuracy')
plt.title('Federated Learning vs. Centralized Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Transparency and Explainability

Machine learning models, especially complex ones, can be difficult to interpret. Explainability techniques help us understand how models make decisions, which is crucial for:

1. Identifying potential biases or errors
2. Building trust with users and stakeholders
3. Meeting regulatory requirements
4. Debugging and improving models

Let's explore some techniques for model explainability:

In [ ]:
# Train a Random Forest model for our loan approval dataset
X = data[['Age', 'Income', 'YearsExperience', 'Education']]
y = data['LoanApproved']

rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X, y)

# 1. Basic feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance)
plt.title('Random Forest Feature Importance', fontsize=15)
plt.tight_layout()
plt.show()

print("Feature importance:")
print(feature_importance)

# 2. Partial dependence plots
from sklearn.inspection import partial_dependence, PartialDependenceDisplay

features = ['Income', 'Age']
plt.figure(figsize=(12, 5))
PartialDependenceDisplay.from_estimator(rf_model, X, features, kind="both")
plt.suptitle('Partial Dependence Plots', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()

In [ ]:
# 3. SHAP values for model explanation
try:
    # Initialize the SHAP explainer
    explainer = shap.TreeExplainer(rf_model)
    
    # Calculate SHAP values for a sample of the data
    sample_indices = np.random.choice(len(X), size=min(100, len(X)), replace=False)
    X_sample = X.iloc[sample_indices]
    shap_values = explainer.shap_values(X_sample)
    
    # Summary plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
    plt.title('SHAP Feature Importance', fontsize=15)
    plt.tight_layout()
    plt.show()
    
    # Individual explanation
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_sample, show=False)
    plt.title('SHAP Feature Values', fontsize=15)
    plt.tight_layout()
    plt.show()
    
    # Force plot for a single prediction
    plt.figure(figsize=(15, 3))
    individual_idx = 0
    shap.initjs()
    shap.force_plot(explainer.expected_value[1], 
                   shap_values[1][individual_idx], 
                   X_sample.iloc[individual_idx],
                   matplotlib=True, 
                   show=False)
    plt.title('SHAP Force Plot (Individual Prediction Explanation)', fontsize=15)
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"SHAP visualization failed: {e}")
    print("Install shap with 'pip install shap' to run this example")

## 6. Accountability and Governance

Accountability in ML involves tracking, documenting, and taking responsibility for decisions made by ML systems. Governance refers to the framework for managing ML development and deployment.

Key elements include:

1. **Model documentation**: Tracking model versions, training data, hyperparameters
2. **Performance monitoring**: Ongoing evaluation of model behavior
3. **Ethical reviews**: Regular assessment of potential negative impacts
4. **Decision audit trails**: Logging model decisions for review
5. **Feedback mechanisms**: Capturing and responding to stakeholder input

Let's create a simple example of model documentation and version control:

In [ ]:
import json
from datetime import datetime
import hashlib
import pickle
import os
from sklearn.model_selection import KFold, cross_val_score

# Create a model card for our loan approval model
def create_model_card(model, X, y, model_name="Loan Approval Model"):
    """Create documentation for a ML model"""
    
    # Calculate model SHA256 hash for version control
    model_bytes = pickle.dumps(model)
    model_hash = hashlib.sha256(model_bytes).hexdigest()
    
    # Calculate data hash
    data_hash = hashlib.sha256(pickle.dumps((X, y))).hexdigest()
    
    # Get cross-validation performance metrics
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_accuracy = cross_val_score(model, X, y, cv=cv, scoring='accuracy').mean()
    
    # Get feature information
    feature_info = {
        'features': list(X.columns),
        'feature_importance': dict(zip(X.columns, model.feature_importances_.tolist()))
    }
    
    # Get bias metrics
    gender = data['Gender']
    gender_acc = {}
    for gender_val in [0, 1]:
        gender_mask = gender == gender_val
        gender_acc[f"Gender_{gender_val}"] = accuracy_score(
            y[gender_mask], 
            model.predict(X[gender_mask])
        )
    
    # Construct model card
    model_card = {
        'model_name': model_name,
        'version': model_hash[:8],  # First 8 chars of hash
        'created_at': datetime.now().isoformat(),
        'created_by': 'Ethics in ML Demo',
        'model_type': model.__class__.__name__,
        'data_hash': data_hash[:8],
        'data_dimensions': {
            'rows': X.shape[0],
            'columns': X.shape[1]
        },
        'performance': {
            'cross_validation_accuracy': float(cv_accuracy),
            'demographic_accuracy': gender_acc
        },
        'features': feature_info,
        'hyperparameters': model.get_params(),
        'intended_use': 'Educational demonstration of ethical ML principles',
        'limitations': 'This is a demo model not meant for production use',
        'ethical_considerations': [
            'Model shows different approval rates between demographic groups',
            'No differential privacy was applied to training data',
            'Application in real-world lending requires additional fairness assessment'
        ]
    }
    
    return model_card

# Create a model card for our random forest model
model_card = create_model_card(rf_model, X, y)

# Print formatted model card
print(json.dumps(model_card, indent=2))

# Example of how to save the model card
model_path = 'loan_approval_model'
try:
    # Create directory if it doesn't exist
    if not os.path.exists(model_path):
        os.makedirs(model_path)
    
    # Save model card as JSON
    with open(os.path.join(model_path, 'model_card.json'), 'w') as f:
        json.dump(model_card, f, indent=2)
    
    # In reality, we'd also save the model here
    # pickle.dump(rf_model, open(os.path.join(model_path, 'model.pkl'), 'wb'))
    
    print(f"\nModel card saved to {model_path}/model_card.json")
except Exception as e:
    print(f"Couldn't save model card due to: {e}")
    print("This is expected in notebook environments with limited file permissions")

## 7. Practical Case Studies

Let's analyze a real-world scenario where ethical ML considerations are important:

### Case Study: Credit Risk Assessment

In this scenario, we're developing a model to predict credit worthiness, similar to our loan approval example above. Let's identify the ethical considerations and potential mitigations:

In [ ]:
# Create a more comprehensive dataset for credit risk assessment
np.random.seed(42)
n = 1000

# Features with potential for disparate impact
age = np.random.normal(40, 12, n)
income = np.random.normal(50000, 20000, n) 
gender = np.random.binomial(1, 0.5, n)  # 0: female, 1: male
race = np.random.choice([0, 1, 2, 3], n, p=[0.7, 0.1, 0.1, 0.1])  # Imbalanced racial distribution
zip_code = np.random.choice(range(10), n, p=[0.05, 0.05, 0.05, 0.05, 0.2, 0.2, 0.1, 0.1, 0.1, 0.1])

# Credit features
credit_score = np.random.normal(700, 100, n)
debt_to_income = np.random.normal(0.3, 0.15, n)
loan_amount = np.random.normal(20000, 10000, n)
loan_term = np.random.choice([36, 60, 84], n)
employment_years = np.random.gamma(5, 1, n)

# Introduce bias: income correlates with gender and race
income = income + gender * 10000  # Gender pay gap
income = income + (race == 0) * 5000  # Racial income disparity

# Credit score correlates with zip code (proxy for redlining)
credit_score = credit_score - (zip_code < 4) * 50

# Create target with bias: default probability
default_prob = (
    0.3  # Base probability
    - 0.001 * credit_score  # Credit score effect
    + debt_to_income  # Debt ratio effect
    - 0.0000005 * income  # Income effect
    + 0.1 * (age < 25)  # Age effect
    + 0.05 * (race != 0)  # Racial bias
)

# Clip probabilities to [0.01, 0.99] range
default_prob = np.clip(default_prob, 0.01, 0.99)
default = np.random.binomial(1, default_prob, n)

# Create DataFrame
credit_data = pd.DataFrame({
    'Age': age,
    'Income': income,
    'Gender': gender,
    'Race': race,
    'ZipCode': zip_code,
    'CreditScore': credit_score,
    'DebtToIncome': debt_to_income,
    'LoanAmount': loan_amount,
    'LoanTerm': loan_term,
    'EmploymentYears': employment_years,
    'Default': default
})

# Map categories
credit_data['GenderCategory'] = credit_data['Gender'].map({0: 'Female', 1: 'Male'})
credit_data['RaceCategory'] = credit_data['Race'].map({0: 'Group A', 1: 'Group B', 
                                                     2: 'Group C', 3: 'Group D'})

# Show data
print("Credit risk assessment dataset:")
print(credit_data.head())

# Analyze default rates by protected attributes
print("\nDefault rate by gender:")
print(credit_data.groupby('GenderCategory')['Default'].mean())

print("\nDefault rate by race:")
print(credit_data.groupby('RaceCategory')['Default'].mean())

print("\nDefault rate by zip code:")
print(credit_data.groupby('ZipCode')['Default'].mean())

# Visualize correlations between features
plt.figure(figsize=(12, 10))
corr = credit_data[['Age', 'Income', 'Gender', 'Race', 'ZipCode', 
                   'CreditScore', 'DebtToIncome', 'Default']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Feature Correlations', fontsize=15)
plt.tight_layout()
plt.show()

# Visualize default rates by protected attributes
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.barplot(x='GenderCategory', y='Default', data=credit_data)
plt.title('Default Rate by Gender')
plt.ylabel('Default Rate')

plt.subplot(1, 3, 2)
sns.barplot(x='RaceCategory', y='Default', data=credit_data)
plt.title('Default Rate by Race')
plt.ylabel('Default Rate')

plt.subplot(1, 3, 3)
sns.barplot(x='ZipCode', y='Default', data=credit_data)
plt.title('Default Rate by Zip Code')
plt.ylabel('Default Rate')

plt.tight_layout()
plt.show()

### Ethical Analysis of Credit Risk Model

Based on our analysis, here are the key ethical concerns and potential mitigations:

**1. Identified Issues:**
- Gender bias in income and default rates
- Racial disparities in default predictions
- Zip code correlations (potential for redlining)
- Imbalanced racial representation in training data

**2. Mitigations:**

Let's implement a more ethical approach to this credit risk assessment:

In [ ]:
# 1. Feature Selection: Remove or carefully handle protected attributes
X_problematic = credit_data[['Age', 'Income', 'Gender', 'Race', 'ZipCode',
                           'CreditScore', 'DebtToIncome', 'LoanAmount', 
                           'LoanTerm', 'EmploymentYears']]

# Remove direct protected attributes and proxies
X_ethical = credit_data[['Age', 'CreditScore', 'DebtToIncome', 'LoanAmount', 
                        'LoanTerm', 'EmploymentYears']]

y = credit_data['Default']

# 2. Compare models with and without problematic features
from sklearn.model_selection import train_test_split

# Split data
X_prob_train, X_prob_test, X_eth_train, X_eth_test, y_train, y_test = train_test_split(
    X_problematic, X_ethical, y, test_size=0.3, random_state=42)

# Train models
problematic_model = RandomForestClassifier(random_state=42, n_estimators=100)
problematic_model.fit(X_prob_train, y_train)

ethical_model = RandomForestClassifier(random_state=42, n_estimators=100)
ethical_model.fit(X_eth_train, y_train)

# Get predictions
y_pred_prob = problematic_model.predict(X_prob_test)
y_pred_eth = ethical_model.predict(X_eth_test)

# 3. Compare overall performance
print("Problematic model accuracy:", accuracy_score(y_test, y_pred_prob))
print("Ethical model accuracy:", accuracy_score(y_test, y_pred_eth))

# 4. Analyze fairness metrics
gender_test = credit_data.loc[y_test.index, 'Gender']
race_test = credit_data.loc[y_test.index, 'Race']

# Create DataFrames with results
results_prob = pd.DataFrame({
    'Gender': gender_test.values,
    'Race': race_test.values,
    'True': y_test.values,
    'Predicted': y_pred_prob
})

results_eth = pd.DataFrame({
    'Gender': gender_test.values,
    'Race': race_test.values,
    'True': y_test.values,
    'Predicted': y_pred_eth
})

# Group results by protected attributes
gender_results_prob = results_prob.groupby('Gender')['Predicted'].mean()
race_results_prob = results_prob.groupby('Race')['Predicted'].mean()

gender_results_eth = results_eth.groupby('Gender')['Predicted'].mean()
race_results_eth = results_eth.groupby('Race')['Predicted'].mean()

# Visualize disparities
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gender disparities
axes[0, 0].bar([0, 1], gender_results_prob, color='crimson')
axes[0, 0].set_title('Problematic Model: Default Rate by Gender')
axes[0, 0].set_xticks([0, 1])
axes[0, 0].set_xticklabels(['Female', 'Male'])
axes[0, 0].set_ylabel('Predicted Default Rate')

axes[0, 1].bar([0, 1], gender_results_eth, color='forestgreen')
axes[0, 1].set_title('Ethical Model: Default Rate by Gender')
axes[0, 1].set_xticks([0, 1])
axes[0, 1].set_xticklabels(['Female', 'Male'])
axes[0, 1].set_ylabel('Predicted Default Rate')

# Race disparities
axes[1, 0].bar(range(4), race_results_prob, color='crimson')
axes[1, 0].set_title('Problematic Model: Default Rate by Race')
axes[1, 0].set_xticks(range(4))
axes[1, 0].set_xticklabels(['Group A', 'Group B', 'Group C', 'Group D'])
axes[1, 0].set_ylabel('Predicted Default Rate')

axes[1, 1].bar(range(4), race_results_eth, color='forestgreen')
axes[1, 1].set_title('Ethical Model: Default Rate by Race')
axes[1, 1].set_xticks(range(4))
axes[1, 1].set_xticklabels(['Group A', 'Group B', 'Group C', 'Group D'])
axes[1, 1].set_ylabel('Predicted Default Rate')

plt.tight_layout()
plt.show()

# Calculate demographic parity differences
gender_diff_prob = abs(gender_results_prob[1] - gender_results_prob[0])
gender_diff_eth = abs(gender_results_eth[1] - gender_results_eth[0])

race_max_diff_prob = max(race_results_prob) - min(race_results_prob)
race_max_diff_eth = max(race_results_eth) - min(race_results_eth)

print("\nGender default rate difference (problematic model):", gender_diff_prob)
print("Gender default rate difference (ethical model):", gender_diff_eth)
print("Race max default rate difference (problematic model):", race_max_diff_prob)
print("Race max default rate difference (ethical model):", race_max_diff_eth)

# Calculate fairness improvement
gender_improvement = (gender_diff_prob - gender_diff_eth) / gender_diff_prob * 100
race_improvement = (race_max_diff_prob - race_max_diff_eth) / race_max_diff_prob * 100

print(f"\nFairness improvement for gender: {gender_improvement:.1f}%")
print(f"Fairness improvement for race: {race_improvement:.1f}%")
print(f"Accuracy impact: {(accuracy_score(y_test, y_pred_eth) - accuracy_score(y_test, y_pred_prob)) * 100:.2f}%")

## Summary: Ethical Considerations in Machine Learning

Machine learning systems can significantly impact people's lives, making ethical considerations essential:

1. **Bias and Fairness**:
   - ML systems can perpetuate or amplify existing social biases
   - Fairness measures (demographic parity, equalized odds) help identify disparities
   - Bias mitigation techniques can reduce unfair treatment of protected groups

2. **Privacy**:
   - ML often requires sensitive personal data
   - Privacy-preserving techniques (anonymization, differential privacy, federated learning) protect individuals
   - Balance must be struck between model utility and privacy

3. **Transparency and Explainability**:
   - Black-box models raise concerns about opacity in decision-making
   - Explainable AI techniques help stakeholders understand model decisions
   - Feature importance, SHAP values, and partial dependence plots increase transparency

4. **Accountability and Governance**:
   - Model cards and documentation capture information about models
   - Performance monitoring ensures continued ethical operation
   - Clear ownership of ML systems is essential for responsible use

5. **Real-world Applications**:
   - Ethical considerations are especially important in high-stakes domains
   - Each domain (healthcare, finance, criminal justice) has unique ethical concerns
   - Continuous ethical review is necessary throughout the ML lifecycle

Remember that ethics in ML is an evolving field, and staying informed about new developments and best practices is crucial for all ML practitioners.

## Resources for Further Learning

1. **Books**:
   - "Weapons of Math Destruction" by Cathy O'Neil
   - "Fairness and Machine Learning" by Solon Barocas, Moritz Hardt, and Arvind Narayanan
   - "The Ethical Algorithm" by Michael Kearns and Aaron Roth

2. **Organizations**:
   - [AI Ethics Guidelines Global Inventory](https://algorithmwatch.org/en/project/ai-ethics-guidelines-global-inventory/)
   - [Partnership on AI](https://www.partnershiponai.org/)
   - [AI Now Institute](https://ainowinstitute.org/)

3. **Tools**:
   - [Fairlearn](https://fairlearn.org/): Toolkit for assessing and improving fairness
   - [AI Fairness 360](https://aif360.mybluemix.net/): Fairness metrics and mitigation algorithms
   - [What-If Tool](https://pair-code.github.io/what-if-tool/): Interactive visual interface for model understanding
   - [SHAP](https://github.com/slundberg/shap): Game theoretic approach to explain model outputs

4. **Courses**:
   - [Ethics of AI](https://ethics-of-ai.mooc.fi/) - University of Helsinki
   - [Practical Data Ethics](https://ethics.fast.ai/) - fast.ai
   - [Fairness in Machine Learning](https://fairmlclass.github.io/) - Various universities